# 01 Data Understanding and Initial Cleaning

## Project Context

This notebook is the first step of the **AI-Powered Lead Generation & Recommendation Analytics Platform** project.

The goal of this stage is to:

1. Load the raw Olist e-commerce datasets;
2. Understand the structure of each table;
3. Perform initial data profiling;
4. Check missing values and duplicates;
5. Prepare the foundation for downstream business analysis, lead scoring, recommendation strategy design, and A/B testing.

Although the original Olist dataset does not contain live-streaming or clickstream data, it provides a strong e-commerce foundation including customers, orders, products, sellers, payments, and reviews. Later in the project, simulated user events such as views, clicks, inquiries, add-to-cart actions, and purchases will be created to support funnel analysis and recommendation strategy evaluation.

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# Define the base project directory.
# Since this notebook is stored inside the notebooks/ folder,
# ".." refers to the project root directory.
BASE_DIR = Path("..")

# Define key data folders
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
FINAL_DIR = BASE_DIR / "data" / "final"

# Define output folders
OUTPUT_DIR = BASE_DIR / "outputs"
TABLE_OUTPUT_DIR = OUTPUT_DIR / "tables"
CHART_OUTPUT_DIR = OUTPUT_DIR / "charts"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "model_results"

# Create output folders if they do not already exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHART_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project paths defined successfully.")

Project paths defined successfully.


In [11]:
# List all CSV files in the raw data folder
files = list(RAW_DIR.glob("*.csv"))

print("CSV files found in data/raw/:")
for file in files:
    print("-", file.name)

print("\nTotal number of CSV files:", len(files))

CSV files found in data/raw/:
- olist_sellers_dataset.csv
- product_category_name_translation.csv
- olist_orders_dataset.csv
- olist_order_items_dataset.csv
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_products_dataset.csv

Total number of CSV files: 9


In [12]:
# Load raw datasets
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

print("All raw datasets loaded successfully.")

All raw datasets loaded successfully.


## Create a Dataset Dictionary

To make the following profiling process more efficient, we store all DataFrames in a Python dictionary.

This allows us to loop through all datasets and generate summary statistics such as row count, column count, duplicate rows, and missing values.

In [13]:
# Store all raw datasets in a dictionary for easier profiling
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

print("Dataset dictionary created successfully.")

Dataset dictionary created successfully.


## Raw Data Summary

This step generates a high-level summary of each raw dataset.This is an important data profiling step because it helps us understand the size, completeness, and potential quality issues of the raw data before cleaning and modelling.

For each table, we calculate:

- Number of rows
- Number of columns
- Number of duplicate rows
- Total number of missing values



In [14]:
summary = []

# Loop through each dataset and calculate basic profiling metrics
for name, df in datasets.items():
    summary.append({
        "table_name": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum()
    })

# Convert the summary list into a DataFrame
raw_summary = pd.DataFrame(summary)

# Display the raw data summary
raw_summary

,table_name,rows,columns,duplicate_rows,missing_values
0,customers,99441,5,0,0
1,geolocation,1000163,5,261831,0
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,0,145903
5,orders,99441,8,0,4908
6,products,32951,9,0,2448
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


In [15]:
# Export raw data summary to outputs/tables/
raw_summary.to_csv(TABLE_OUTPUT_DIR / "raw_data_summary.csv", index=False)

print("Raw data summary exported successfully.")

Raw data summary exported successfully.


## Inspect Column Names

Next, we inspect the column names of each dataset. Understanding table structures is essential before building the analytical data model.

This step helps us understand:

1. Which columns can be used as primary keys or foreign keys;
2. Which fields are useful for user, product, seller, order, payment, and review analysis;
3. Which fields need to be renamed or cleaned;
4. Which missing business fields may need to be simulated later, such as views, clicks, inquiries, and add-to-cart events.



In [16]:
# Print column names for each dataset
for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    print(df.columns.tolist())


===== CUSTOMERS =====
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

===== GEOLOCATION =====
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

===== ORDER_ITEMS =====
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

===== PAYMENTS =====
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

===== REVIEWS =====
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

===== ORDERS =====
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

===== PRODUCTS =====
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'prod

## Missing Value Analysis

This step calculates missing values at the column level.

For each column with missing values, we calculate:

- Missing value count
- Missing value rate

This analysis helps us decide how to handle missing data during the cleaning process. For example, missing review comments may not be a data quality issue because many customers leave a score without writing text. However, missing delivery dates may be important for logistics or customer satisfaction analysis.

In [17]:
missing_summary = []

# Loop through each dataset to calculate missing value statistics
for name, df in datasets.items():
    missing_count = df.isna().sum()
    missing_rate = df.isna().mean()

    temp = pd.DataFrame({
        "table_name": name,
        "column_name": missing_count.index,
        "missing_count": missing_count.values,
        "missing_rate": missing_rate.values
    })

    # Keep only columns with missing values
    temp = temp[temp["missing_count"] > 0]

    missing_summary.append(temp)

# Combine all missing value summaries into one DataFrame
missing_summary = pd.concat(missing_summary, ignore_index=True)

# Sort by missing rate from high to low
missing_summary = missing_summary.sort_values(by="missing_rate", ascending=False)

missing_summary

,table_name,column_name,missing_count,missing_rate
0,reviews,review_comment_title,87656,0.883415
1,reviews,review_comment_message,58247,0.587025
4,orders,order_delivered_customer_date,2965,0.029817
5,products,product_category_name,610,0.018512
6,products,product_name_lenght,610,0.018512
7,products,product_description_lenght,610,0.018512
8,products,product_photos_qty,610,0.018512
3,orders,order_delivered_carrier_date,1783,0.017930
2,orders,order_approved_at,160,0.001609
9,products,product_weight_g,2,0.000061


In [18]:
# Export missing value summary to outputs/tables/
missing_summary.to_csv(TABLE_OUTPUT_DIR / "missing_value_summary.csv", index=False)

print("Missing value summary exported successfully.")

Missing value summary exported successfully.


## Initial Observations

Based on the raw data profiling and missing value analysis, we can make several initial observations:

1. The Olist dataset contains multiple relational tables covering customers, orders, products, sellers, payments, and reviews.
2. The dataset is suitable for building an e-commerce analytics workflow.
3. Some fields contain missing values, especially review text and delivery-related timestamps.
4. Missing review text does not necessarily mean invalid data because users may provide a rating without writing a comment.
5. The original dataset does not include clickstream or live-stream interaction events, so simulated user event logs will be created in a later step.
6. The next step is to join the core business tables and create a cleaned order-level analytical dataset.

This initial profiling step provides the foundation for downstream SQL analysis, funnel analysis, lead scoring, recommendation strategy design, and A/B testing.

## Prepare Product Category Translation

The original Olist product categories are in Portuguese. To make the analysis more readable and suitable for business reporting, we join the product table with the category translation table.

This step creates an English product category field that will be used in later KPI analysis, funnel analysis, recommendation strategy design, and dashboard visualisation.

In [19]:
# Join product table with English category translation
products_en = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

# Preview translated product table
products_en.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


## Aggregate Payment Data at Order Level

The payments table may contain multiple payment records for the same order.  
For example, one order may be paid using multiple payment methods or multiple installments.

To create an order-level analytical dataset, we aggregate payment information by `order_id`.

For each order, we calculate:

- Total payment value
- Average number of payment installments
- Main payment type

In [20]:
# Aggregate payment information at order level
payment_agg = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "mean"),
    payment_type=("payment_type", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
).reset_index()

# Preview aggregated payment table
payment_agg.head()

,order_id,payment_value,payment_installments,payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2.0,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3.0,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5.0,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2.0,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3.0,credit_card


## Aggregate Review Data at Order Level

The reviews table contains customer feedback, including review scores and review text.

For the lead generation and LLM intent analysis part of this project, review text is important because it can be used to infer customer sentiment and purchase-related intent.

In this step, we aggregate review information at the order level.

In [21]:
# Aggregate review information at order level
review_agg = reviews.groupby("order_id").agg(
    review_score=("review_score", "mean"),
    review_comment_title=("review_comment_title", "first"),
    review_comment_message=("review_comment_message", "first")
).reset_index()

# Preview aggregated review table
review_agg.head()

,order_id,review_score,review_comment_title,review_comment_message
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,None,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,4.0,None,None
2,000229ec398224ef6ca0657da4fc703e,5.0,None,Chegou antes do prazo previsto e o produto sur...
3,00024acbcdf0a6daa1e931b038114c75,4.0,None,None
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,None,Gostei pois veio no prazo determinado .


In [22]:
# Convert order timestamp columns into datetime format
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

print("Date columns converted successfully.")

Date columns converted successfully.


## Build the Core Order-Level Analytical Dataset

This step creates the main analytical dataset by joining:

- Order items
- Orders
- Customers
- Products
- Sellers
- Payments
- Reviews

The output dataset provides a unified view of each transaction, including user information, product information, seller information, payment information, and review feedback.

This table will be the foundation for the rest of the project.

In [23]:
# Build the core analytical dataset
order_base = (
    order_items
    .merge(orders, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(products_en, on="product_id", how="left")
    .merge(sellers, on="seller_id", how="left")
    .merge(payment_agg, on="order_id", how="left")
    .merge(review_agg, on="order_id", how="left")
)

# Preview the merged dataset
order_base.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,review_score,review_comment_title,review_comment_message
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,cool_stuff,27277,volta redonda,SP,72.19,2.0,credit_card,5.0,None,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,pet_shop,3471,sao paulo,SP,259.83,3.0,credit_card,4.0,None,None
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,furniture_decor,37564,borda da mata,MG,216.87,5.0,credit_card,5.0,None,Chegou antes do prazo previsto e o produto sur...
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,perfumery,14403,franca,SP,25.78,2.0,credit_card,4.0,None,None
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,garden_tools,87900,loanda,PR,218.04,3.0,credit_card,5.0,None,Gostei pois veio no prazo determinado .


## Select and Rename Key Fields

The merged dataset contains many columns.  
To make the dataset easier to use for analysis and modelling, we select only the fields that are relevant to this project.

We also rename several columns to improve readability.

For example:

- `customer_unique_id` is renamed to `user_id`
- `order_purchase_timestamp` is renamed to `order_time`
- `product_category_name_english` is renamed to `category`

The original dataset also contains spelling mistakes such as `product_name_lenght`.  
We rename them to the correct form, such as `product_name_length`.

In [24]:
# Select project-relevant fields
order_base_clean = order_base[[
    "order_id",
    "customer_id",
    "customer_unique_id",
    "product_id",
    "seller_id",
    "order_purchase_timestamp",
    "order_status",
    "price",
    "freight_value",
    "payment_value",
    "payment_type",
    "payment_installments",
    "product_category_name_english",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "seller_city",
    "seller_state",
    "customer_city",
    "customer_state",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].copy()

# Rename columns for readability and consistency
order_base_clean.rename(columns={
    "customer_unique_id": "user_id",
    "order_purchase_timestamp": "order_time",
    "product_category_name_english": "category",
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
}, inplace=True)

# Preview cleaned column structure
order_base_clean.head()

,order_id,customer_id,user_id,product_id,seller_id,order_time,order_status,price,freight_value,payment_value,...,product_photos_qty,seller_city,seller_state,customer_city,customer_state,review_score,review_comment_title,review_comment_message,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-13 08:59:02,delivered,58.90,13.29,72.19,...,4.0,volta redonda,SP,campos dos goytacazes,RJ,5.0,None,"Perfeito, produto entregue antes do combinado.",2017-09-20 23:43:48,2017-09-29
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-04-26 10:53:06,delivered,239.90,19.93,259.83,...,2.0,sao paulo,SP,santa fe do sul,SP,4.0,None,None,2017-05-12 16:04:24,2017-05-15
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-14 14:33:31,delivered,199.00,17.87,216.87,...,2.0,borda da mata,MG,para de minas,MG,5.0,None,Chegou antes do prazo previsto e o produto sur...,2018-01-22 13:19:16,2018-02-05
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-08 10:00:35,delivered,12.99,12.79,25.78,...,1.0,franca,SP,atibaia,SP,4.0,None,None,2018-08-14 13:32:39,2018-08-20
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-04 13:57:51,delivered,199.90,18.14,218.04,...,1.0,loanda,PR,varzea paulista,SP,5.0,None,Gostei pois veio no prazo determinado .,2017-03-01 16:42:31,2017-03-17


## Initial Data Cleaning

This step applies basic cleaning rules to the core analytical dataset.

Cleaning decisions:

1. Remove rows with missing key identifiers, because they cannot be reliably joined or analysed.
2. Keep only rows with valid positive product prices.
3. Fill missing product categories with `unknown`.
4. Fill missing review scores with the median review score.
5. Create date-related fields for time-series analysis.
6. Create a GMV field as product price plus freight value.

GMV stands for Gross Merchandise Value.  
In this project, it is used as a proxy for transaction value.

In [25]:
# Remove rows with missing key identifiers
order_base_clean = order_base_clean.dropna(subset=[
    "order_id", "user_id", "product_id", "seller_id"
])

# Keep only valid positive product prices
order_base_clean = order_base_clean[order_base_clean["price"] > 0]

# Fill missing product category
order_base_clean["category"] = order_base_clean["category"].fillna("unknown")

# Fill missing review score using median value
order_base_clean["review_score"] = order_base_clean["review_score"].fillna(
    order_base_clean["review_score"].median()
)

# Create date fields
order_base_clean["order_date"] = order_base_clean["order_time"].dt.date
order_base_clean["order_year"] = order_base_clean["order_time"].dt.year
order_base_clean["order_month"] = order_base_clean["order_time"].dt.to_period("M").astype(str)

# Create GMV field
order_base_clean["gmv"] = order_base_clean["price"] + order_base_clean["freight_value"]

# Preview cleaned dataset information
order_base_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 29 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  object        
 1   customer_id                    112650 non-null  object        
 2   user_id                        112650 non-null  object        
 3   product_id                     112650 non-null  object        
 4   seller_id                      112650 non-null  object        
 5   order_time                     112650 non-null  datetime64[ns]
 6   order_status                   112650 non-null  object        
 7   price                          112650 non-null  float64       
 8   freight_value                  112650 non-null  float64       
 9   payment_value                  112647 non-null  float64       
 10  payment_type                   112647 non-null  object        
 11  

## Create Delivery Delay Features

Delivery experience can affect customer satisfaction and review scores.

In this step, we create delivery-related features:

- Estimated delivery date
- Actual delivery date
- Delivery delay in days
- Late delivery flag

These fields can later be used to analyse whether delayed delivery affects review scores and customer satisfaction.

In [26]:
# Calculate delivery delay in days
order_base_clean["delivery_delay_days"] = (
    order_base_clean["order_delivered_customer_date"] 
    - order_base_clean["order_estimated_delivery_date"]
).dt.days

# Create late delivery flag
# 1 means the order was delivered later than the estimated delivery date
# 0 means the order was delivered on time or earlier
order_base_clean["late_delivery_flag"] = np.where(
    order_base_clean["delivery_delay_days"] > 0, 1, 0
)

# Preview delivery-related fields
order_base_clean[[
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delay_days",
    "late_delivery_flag",
    "review_score"
]].head()

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,late_delivery_flag,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-20 23:43:48,2017-09-29,-9.0,0,5.0
1,00018f77f2f0320c557190d7a144bdd3,2017-05-12 16:04:24,2017-05-15,-3.0,0,4.0
2,000229ec398224ef6ca0657da4fc703e,2018-01-22 13:19:16,2018-02-05,-14.0,0,5.0
3,00024acbcdf0a6daa1e931b038114c75,2018-08-14 13:32:39,2018-08-20,-6.0,0,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-03-01 16:42:31,2017-03-17,-16.0,0,5.0


## Create Price Band Feature

Price band is useful for product segmentation and recommendation analysis.

In this step, products are grouped into four price bands:

- Low
- Medium
- High
- Premium

This feature can help analyse whether different price levels have different conversion patterns, review scores, or recommendation performance.

In [27]:
# Create price band based on product price quartiles
order_base_clean["price_band"] = pd.qcut(
    order_base_clean["price"],
    q=4,
    labels=["Low", "Medium", "High", "Premium"],
    duplicates="drop"
)

# Preview price band distribution
order_base_clean["price_band"].value_counts()

price_band
Low        28501
Premium    28114
High       28095
Medium     27940
Name: count, dtype: int64

In [28]:
# Final quality check
print("Final dataset shape:", order_base_clean.shape)
print("Duplicate rows:", order_base_clean.duplicated().sum())
print("Total missing values:", order_base_clean.isna().sum().sum())

# Display descriptive statistics
order_base_clean[[
    "price",
    "freight_value",
    "payment_value",
    "gmv",
    "review_score",
    "delivery_delay_days"
]].describe()

Final dataset shape: (112650, 32)
Duplicate rows: 10225
Total missing values: 174137


,price,freight_value,payment_value,gmv,review_score,delivery_delay_days
count,112650.000000,112650.000000,112647.000000,112650.000000,112650.000000,110196.000000
mean,120.653739,19.990320,180.281186,140.644059,4.041213,-12.030201
std,183.633928,15.806405,272.849042,190.724394,1.383116,10.160157
min,0.850000,0.000000,9.590000,6.080000,1.000000,-147.000000
25%,39.900000,13.080000,65.670000,55.220000,4.000000,-17.000000
50%,74.990000,16.260000,114.440000,92.320000,5.000000,-13.000000
75%,134.900000,21.150000,195.390000,157.937500,5.000000,-7.000000
max,6735.000000,409.680000,13664.080000,6929.310000,5.000000,188.000000


In [29]:
# Export cleaned order-level analytical dataset
order_base_clean.to_csv(PROCESSED_DIR / "clean_order_base.csv", index=False)

print("Cleaned order-level analytical dataset exported successfully.")
print("File saved to:", PROCESSED_DIR / "clean_order_base.csv")

Cleaned order-level analytical dataset exported successfully.
File saved to: ../data/processed/clean_order_base.csv
